# 08 Fig. 2 and the form assignment

The index is built from effective free-flow speed alone (constants of the single fit, `MFDs_Curve_uncertainty/point_estimate.json`), so it is equivalent to a switch speed u*; the figure places networks by u_fe and marks u*. "Correct" means the index picks the branch of eq. (5) that fits the network better, the definition the cross-validation uses. The notebook also scores the standard cubic and Underwood forms, for the text.


In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "scripts")))
from paths import MFD_DIR, NODES_DIR, OUTPUTS_DIR, RESULTS

import json, os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT = os.path.join(RESULTS, "Fig2_uf_index")
os.makedirs(OUT, exist_ok=True)

import mfd_uncertainty as mu   # the same code that produced the fit and the cross-validation

pe = json.load(open(os.path.join(RESULTS, "MFDs_Curve_uncertainty", "point_estimate.json")))
hp = pe["hybrid_params"]
assert hp["w_L"] == hp["w_SL"] == hp["w_C"] == 0.0, "expected the free-flow-speed-only index"
theta = np.array([hp[n] for n in mu.HYBRID_NAMES], float)
a, s_K, d = hp["a"], hp["s_K"], hp["d"]

prep = mu.build_prepared(MFD_DIR, OUTPUTS_DIR, NODES_DIR, nbins=400,
                         kmax_factor=mu.KMAX_FACTOR_DEFAULT, keep_exact=True, quiet=True)
n = prep.n_cities
Z, mom = mu.build_ui_features(prep.metrics, np.ones(n))
ui = mu.ui_from_Z(Z, theta[4], theta[5:9])

# the stored per-network index must be the same numbers
stored = pd.read_csv(os.path.join(RESULTS, "MFDs_Curve_uncertainty", "per_network_UI.csv")).set_index("city")
assert np.allclose(stored.loc[prep.cities, "UI"].to_numpy(), ui, atol=1e-9)

# the switch speed u*: UI = b + w (exp(clip(z, -2, 2)) - mu_phi) / sd_phi = 0, z = (u - mu_u) / sd_u
m = pe["ui_moments"]
e_star = m["phi_mean"][0] - hp["b"] * m["phi_std"][0] / hp["w_uf"]
z_star = np.log(e_star)
assert hp["w_uf"] > 0 and -2 < z_star < 2, "switch speed outside the unclipped range"
u_star = m["metric_mean"][0] + m["metric_std"][0] * z_star
assert np.array_equal(ui <= mu.U_TH_FIXED, prep.uf <= u_star), "UI <= 0 must equal u_fe <= u*"
print(f"switch speed u* = {u_star:.2f} km/h")

# branch errors, exact (unbinned) and binned (as the cross-validation scores them)
res = {}
for kind, scorer in (("exact", mu._nrmse_per_city_exact), ("binned", mu._nrmse_per_city_binned)):
    nr_cub = scorer(prep, np.zeros(n), a, d, s_K)
    nr_exp = scorer(prep, np.ones(n), a, d, s_K)
    better_exp = nr_exp < nr_cub
    pred_exp = ui <= mu.U_TH_FIXED
    res[kind] = dict(nr_cub=nr_cub, nr_exp=nr_exp, better_exp=better_exp,
                     acc=float(np.mean(pred_exp == better_exp)))
    print(f"[{kind}] form assignment by the index: {res[kind]['acc']:.4f} "
          f"({int(np.sum(pred_exp == better_exp))}/{n}); exponential better for "
          f"{int(better_exp.sum())}; index says urbanized for {int(pred_exp.sum())}")

r = res["exact"]
df = pd.DataFrame({"city": prep.cities, "uf": prep.uf, "UI": ui,
                   "nrmse_cubic_branch": r["nr_cub"], "nrmse_exp_branch": r["nr_exp"],
                   "better_branch": np.where(r["better_exp"], "exponential", "cubic"),
                   "index_says": np.where(ui <= mu.U_TH_FIXED, "exponential", "cubic")})
df["correct"] = df["better_branch"] == df["index_says"]
df["best_nrmse"] = np.minimum(df["nrmse_cubic_branch"], df["nrmse_exp_branch"])
df.to_csv(os.path.join(OUT, "fig2_per_network.csv"), index=False)

def thr_acc(uf, better_exp, t):
    return float(np.mean((uf <= t) == better_exp))
grid = np.unique(np.round(prep.uf, 6))
best_t = max(grid, key=lambda t: thr_acc(prep.uf, r["better_exp"], t))
summary = {
    "a": a, "d": d, "s_u": hp["s_u"], "s_K": s_K, "b": hp["b"], "w_uf": hp["w_uf"],
    "u_star_kmh": float(u_star), "n_networks": n,
    "assignment_index_exact": res["exact"]["acc"], "assignment_index_binned": res["binned"]["acc"],
    "n_exponential_better": int(r["better_exp"].sum()),
    "n_index_urbanized": int((ui <= mu.U_TH_FIXED).sum()),
    "uf_40kmh_vs_branches": thr_acc(prep.uf, r["better_exp"], 40.0),
    "uf_best_threshold_vs_branches": [float(best_t), thr_acc(prep.uf, r["better_exp"], best_t)],
}

# against the two standard forms (cubic a = 0.157, Underwood d = 1), as in Extended Data Fig. 4
e = pd.read_csv(os.path.join(RESULTS, "MFDs_FINAL", "per_city_errors_normalized.csv")).set_index("city").loc[prep.cities]
und_better = (e["RMSE_Underwood"] < e["RMSE_Cubic"]).to_numpy()
summary["n_underwood_better_standard_forms"] = int(und_better.sum())
summary["assignment_index_vs_standard_forms"] = float(np.mean((ui <= 0) == und_better))
summary["uf_40kmh_vs_standard_forms"] = thr_acc(prep.uf, und_better, 40.0)
summary["agreement_branch_vs_standard_labels"] = float(np.mean(und_better == r["better_exp"]))
json.dump(summary, open(os.path.join(OUT, "fig2_summary.json"), "w"), indent=1)
print(json.dumps(summary, indent=1))

# ---- figure, in the style of the published Fig. 2
fig, ax = plt.subplots(figsize=(10, 6), dpi=216)
for lab, col in (("cubic", "tab:blue"), ("exponential", "tab:green")):
    s = df[df["better_branch"] == lab]
    ax.scatter(s["uf"], s["best_nrmse"], s=70, c=col, edgecolors="black", linewidths=0.6,
               label=lab.capitalize(), zorder=3)
ax.axvline(u_star, color="0.35", linestyle="--", linewidth=1.0, zorder=2)
ax.set_xlabel("Effective free-flow speed (km/h)", fontsize=13)
ax.set_ylabel("RMSE of the better branch (normalized)", fontsize=13)
ax.set_xlim(0, max(115, float(df["uf"].max()) + 5))
ax.tick_params(labelsize=12)
ax.grid(True, alpha=0.35)
ax.legend(title="Better branch of eq. (5)", fontsize=12, title_fontsize=12, loc="upper right")
fig.tight_layout()
fig.savefig(os.path.join(OUT, "fig2_uf_index.png"), bbox_inches="tight")
plt.show()
